In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import solve_ivp

# 1. 4次元ロトカ・ヴォルテラ系の定義
def lv_4species(t, state, rx, ry, L, C, D):
    x1, x2, y1, y2 = state
    
    # 被食者の式
    dx1dt = x1 * (rx[0] - L[0,0]*y1 - L[0,1]*y2)
    dx2dt = x2 * (rx[1] - L[1,0]*y1 - L[1,1]*y2)
    
    # 捕食者の式
    dy1dt = y1 * (-ry[0] + C[0]*L[0,0]*x1 + D[0]*L[1,0]*x2)
    dy2dt = y2 * (-ry[1] + C[1]*L[0,1]*x1 + D[1]*L[1,1]*x2)
    
    return [dx1dt, dx2dt, dy1dt, dy2dt]

# 2. パラメータ設定 (カオスや複雑な振動が出やすい値を仮定)
rx = np.array([1.5, 1.0])    # 被食者の増殖率
ry = np.array([1.0, 1.0])    # 捕食者の死亡率
L = np.array([[0.5, 0.4],    # 摂食レート (l11, l12, l21, l22)
              [0.2, 0.5]])
C = np.array([0.8, 0.8])    # 変換効率 c1, c2
D = np.array([0.8, 0.8])    # 変換効率 d1, d2

# 初期値と時間設定
y0 = [2.0, 2.0, 1.5, 1.5]
t_span = (0, 5000)
t_eval = np.linspace(*t_span, 200000)

# 3. ポアンカレ断面の設定
# ここでは「x1 = 1.0」を断面とし、増加方向に横切る瞬間を記録します
# 変更前（エラーになる）
# def poincare_event(t, state):

# 変更後（これで動きます）
def poincare_event(t, state, *args):
    return state[0] - 2.0

poincare_event.terminal = False
poincare_event.direction = 1  # 負から正へ抜ける時のみ

# 4. 数値計算の実行
sol = solve_ivp(
    lv_4species, t_span, y0, args=(rx, ry, L, C, D),
    events=poincare_event, t_eval=t_eval, rtol=1e-8, atol=1e-10
)

# 5. データの抽出 (過渡状態を除去)
points = sol.y_events[0]
stable_points = points[len(points)//5:]  # 最初の20%を捨てる

x2_cross = stable_points[:, 1]
y1_cross = stable_points[:, 2]
y2_cross = stable_points[:, 3]

# 6. 可視化
fig = plt.figure(figsize=(15, 5))

# --- ポアンカレ断面図 (y1 vs y2) ---
ax1 = fig.add_subplot(131)
ax1.scatter(y1_cross, y2_cross, s=2, c='darkgreen', alpha=0.6)
ax1.set_title("Poincaré Section\n($y_1$ vs $y_2$ at $x_1=1.0$)")
ax1.set_xlabel("$y_1$")
ax1.set_ylabel("$y_2$")

# --- リターンマップ (y1_n vs y1_{n+1}) ---
ax2 = fig.add_subplot(132)
ax2.scatter(y1_cross[:-1], y1_cross[1:], s=2, c='blue', alpha=0.6)
ax2.set_title("Return Map\n($y_{1,n}$ vs $y_{1,n+1}$)")
ax2.set_xlabel("$y_{1,n}$")
ax2.set_ylabel("$y_{1,n+1}$")

# --- 時系列の確認 ---
ax3 = fig.add_subplot(133)
ax3.plot(sol.t[-500:], sol.y[0, -500:], label='$x_1$')
ax3.axhline(1.0, color='red', linestyle='--', alpha=0.5, label='Section')
ax3.set_title("Time Series (Last 500 steps)")
ax3.legend()

plt.tight_layout()
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import odeint
import sympy

def find_equilibrium_points_numeric(params):
    """パラメータ辞書から平衡点を数値的に計算する"""
    x1, y1, x2, y2 = sympy.symbols('x1 y1 x2 y2')
    
    r_x1, r_y1, r_x2, r_y2 = params['r']
    l11, l12 = params['lambda'][0]
    l21, l22 = params['lambda'][1]
    c1, c2 = params['c']
    d1, d2 = params['d']

    # 方程式系 = 0
    eqs = [
        r_x1*x1 - l11*x1*y1 - l12*x1*y2,
        -r_y1*y1 + c1*l11*x1*y1 + d1*l21*x2*y1,
        r_x2*x2 - l21*x2*y1 - l22*x2*y2,
        -r_y2*y2 + c2*l12*x1*y2 + d2*l22*x2*y2
    ]

    # 解を求める
    sols = sympy.solve(eqs, (x1, y1, x2, y2))
    
    # 全ての種が生存(>0)している、または意味のある正の解を抽出
    valid_sols = []
    for sol in sols:
        if all(val.is_real and val >= 0 for val in sol):
            valid_sols.append([float(val) for val in sol])
    
    return valid_sols

# 1. モデル方程式の定義
def model(z, t, params):
    x1, y1, x2, y2 = z
    r_x1, r_y1, r_x2, r_y2 = params['r']
    lam = params['lambda']
    c, d = params['c'], params['d']
    l11, l12 = lam[0]
    l21, l22 = lam[1]
    c1, c2 = c
    d1, d2 = d
    
    dx1dt = r_x1*x1 - l11*x1*y1 - l12*x1*y2
    dy1dt = -r_y1*y1 + c1*l11*x1*y1 + d1*l21*x2*y1
    dx2dt = r_x2*x2 - l21*x2*y1 - l22*x2*y2
    dy2dt = -r_y2*y2 + c2*l12*x1*y2 + d2*l22*x2*y2
    
    return [dx1dt, dy1dt, dx2dt, dy2dt]

# 2. パラメータ設定
params = {
    'r': [1.5, 1.0, 1.0, 1.0],
    'lambda': [[0.5, 0.3],
               [0.3, 0.5]],
    'c': [0.8, 0.8],
    'd': [0.8, 0.8]
}

# --- 平衡点の計算 ---
all_equilibria = find_equilibrium_points_numeric(params)
print("見つかった平衡点一覧:")
for i, pt in enumerate(all_equilibria):
    print(f"Point {i}: x1={pt[0]:.3f}, y1={pt[1]:.3f}, x2={pt[2]:.3f}, y2={pt[3]:.3f}")

# 共存点（すべての値が0より大きいもの）を初期値に選ぶ
# もし共存点がなければ、リストの最後の解を使用する
z0 = all_equilibria[-1] 

# 安定性を確認するために、少しだけ平衡点からずらす（オプション）
# z0 = [v + 0.1 for v in z0] 

# 時間軸
t = np.linspace(0, 150, 50000)

# 3. 微分方程式を解く
z = odeint(model, z0, t, args=(params,))

# --- 可視化 (元コードを維持) ---
x1, y1, x2, y2 = z[:, 0], z[:, 1], z[:, 2], z[:, 3]

plt.figure(figsize=(14, 6))
# (以下、グラフ描画処理はご提示のコードと同様)
plt.subplot(1, 2, 1)
plt.plot(t, x1, label='$x_1$ (Prey 1)')
plt.plot(t, y1, label='$y_1$ (Predator 1)', alpha=0.7)
plt.plot(t, x2, label='$x_2$ (Prey 2)', linestyle='--')
plt.plot(t, y2, label='$y_2$ (Predator 2)', alpha=0.7, linestyle='--')
plt.xlabel('Time'); plt.ylabel('Population'); plt.legend(); plt.grid(True)

plt.subplot(1, 2, 2)
plt.plot(x1, y1, label='Pair 1 ($x_1, y_1$)', color='blue')
plt.plot(x2, y2, label='Pair 2 ($x_2, y_2$)', color='red', linestyle='--')
plt.xlabel('Prey'); plt.ylabel('Predator'); plt.legend(); plt.grid(True)

plt.show()